### Setup
Autoreload and imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Imports
import sys
from pathlib import Path
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent, SaliencyMapMethod, CarliniL2Method

sys.path.append(str(Path.cwd().parents[2]))

from utils.functions import get_windowed_data
from utils.notebook import get_model_classifier, clean_data_test, adv_test, FilenameLoader

/home/sliu/Documents/GitHub/reu26/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Define Inputs

In [3]:
## Inputs
checkpoint_name, data_name, save_name = FilenameLoader.rand_pos()

checkpoint_file= f"../../../saved_models/{checkpoint_name}"
data_file = f"../../../data/{data_name}"
save_path = f"../final_data/{save_name}"

collapsed = False # True for JSMA and CW
save_clean = False

### Load Model and Data
Load the data and model and run on clean data to check the accuracy/F1.

In [4]:
## Load model and data
# Load original model and ART-wrapper classifier
model, classifier = get_model_classifier(checkpoint_file, collapsed)

# Load data
(x_train, y_train), (x_test, y_test), fed_dataset, scaler = get_windowed_data(data_file, 
                                                                      normalize=True, 
                                                                      train_perc=80)

GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
/home/sliu/Documents/GitHub/reu26/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Checkpoint path exists!


In [5]:
## Run clean data test
out = clean_data_test(model = model, classifier = classifier, # model information
                x_test=x_test, y_test=y_test, # data information
                checkpoint_file=checkpoint_file, data_file=data_file, # to save in json
                save_path=save_path, filename="clean.json", # saving information
                save_results=save_clean,
                collapsed=collapsed
                ) 

# Check if output passes
print('noWrapper:', 'PASS' if out['noWrapper']['f1'] > 0.8 else 'FAIL')
print('wrapper:', 'PASS' if out['wrapper']['f1'] > 0.8 else 'FAIL')

save=False, Metrics not saved
Metrics {'noWrapper': {'accuracy': 0.9863296840687964, 'recall': 0.9547720528729431, 'precision': 0.9991784882489942, 'f1': 0.976470669379591, 'falseNegativeRate': 0.04522794712705692, 'falsePositiveRate': 0.0003317978655477515, 'TP': 353934, 'TN': 876749, 'FP': 291, 'FN': 16766}, 'wrapper': {'accuracy': 0.9863296840687964, 'precision': 0.9991784882489942, 'recall': 0.9547720528729431, 'f1': 0.976470669379591, 'falseNegativeRate': 0.04522794712705692, 'falsePositiveRate': 0.0003317978655477515, 'TP': 353934, 'TN': 876749, 'FP': 291, 'FN': 16766}, 'files': {'checkpointFile': '../../../saved_models/RandomPos-final.ckpt', 'dataFile': '../../../data/RandomPos_0709.csv'}}
noWrapper: PASS
wrapper: PASS


### Adversarial Test
Run adversarial attacks and save the outputted metrics

In [ ]:
# running: rand speed

for i in range(1, 31):
    eps = float(i/100)
    adv_test(
        classifier = classifier, # classifier
        x_test=x_test, y_test=y_test, # test data
        checkpoint_file=checkpoint_file, data_file=data_file, # for json
        
        end_index = len(y_test.numpy()),
        path = f"{save_path}/pgd-5000",
        filename=f"adv_eps_{eps}.json",
        collapsed=collapsed,
        Attack=ProjectedGradientDescent,
        eps=eps, # kwargs
        targeted=False
        )

=== Attack: ProjectedGradientDescent, kwargs: {'eps': 0.01, 'targeted': False} ===


PGD - Batches:   0%|          | 0/157 [00:00<?, ?it/s]